# Preliminary Analysis

This notebook shows a preliminiary analysis of the synthetic system.

In [1]:
using DelimitedFiles
using PeriodicOrbitTTV

using JLD2

using PyPlot
using LaTeXStrings
using Distributions, LinearAlgebra, StatsBase

## Load the synthetic system parameters

In [2]:
clean_params = vec(readdlm("../frequentist/sample_orbit_v1.in"))

# Wrap the angles such that they are within [-pi, pi]
clean_params[4:8] .= rem2pi.(clean_params[4:8], RoundNearest)

5-element view(::Vector{Float64}, 4:8) with eltype Float64:
 -1.4460348444731441
 -0.7298037482132292
  2.766617650123391
 -3.134479044794369
 -3.1352006973471744

In [3]:
# Initialize the orbital parameters with observation baseline of 500 days.
orbparams = OrbitParameters(3, [0.5], 500.)

# Initialize the orbital object 
optparams_1 = OptimParameters(orbparams.nplanet, clean_params)
orbit_1 = Orbit(orbparams.nplanet, optparams_1, orbparams)

Orbit
Number of planets: 3
Current time: 0.0
Planet masses: [4.6700000000256675e-5, 2.229999999789323e-5, 7.640000000029858e-5]


In [4]:
"""Compute the angle for which the longitude of periastron of the innermost planet shifts."""
function diff_angle_jac(init_state, final_state, ic)
    shifting_angle = get_Jacobi_orbital_elements(final_state, ic)[2].ω - get_Jacobi_orbital_elements(init_state, ic)[2].ω
    shifting_angle = rem2pi(shifting_angle, RoundNearest)
    return shifting_angle
end

diff_angle_jac

## Compute the shifted ω

In [5]:
# Compute the shifted ω per one system periond
shifted_omega = diff_angle_jac(orbit_1.s, orbit_1.state_final, orbit_1.ic)

-0.0530335246778473

In [6]:
# Compute the shifted ω in degrees per year
t_sys = clean_params[end]
shifted_omega / (t_sys) * 180/pi

-0.06269491897178597

## Compute TTV Super-periods

In [7]:
# Calculate Jacobi periods in days for each homework
periods = Vector{Float64}(undef, 3)

for i in 2:4
    @show periods[i-1] = get_Jacobi_orbital_elements(orbit_1.s, orbit_1.ic)[i].P
end

periods[i - 1] = ((get_Jacobi_orbital_elements(orbit_1.s, orbit_1.ic))[i]).P = 12.142000000000007
periods[i - 1] = ((get_Jacobi_orbital_elements(orbit_1.s, orbit_1.ic))[i]).P = 24.33860933474536
periods[i - 1] = ((get_Jacobi_orbital_elements(orbit_1.s, orbit_1.ic))[i]).P = 48.87943881771109


In [8]:
# Compute delta
delta = (periods[2]/periods[1])*(1/2) - 1

periods[2]/(2*delta)

5411.517937739897

In [9]:
delta = (periods[3]/periods[2])*(1/2) - 1

periods[3]/(2*delta)

5882.982365285658

From the preliminary results, it looks like both pairs of planet have different TTV super-periods. We shall use the actual signals to calculate the super-periods later instead.